In [1]:
!pip install numba

In [2]:
import numpy as np
import pandas as pd
from numba import njit
import csv
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.metrics import root_mean_squared_error

In [3]:
input_files = [
    "/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_1.txt",
    "/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_2.txt",
    "/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_3.txt",
    "/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_4.txt"
]

output_file = "netflix_ratings.csv"

with open(output_file, "w", newline="") as out:
    writer = csv.writer(out)
    writer.writerow(["customer_id", "movie_id", "rating", "date"])

    for file in input_files:
        with open(file, "r") as f:
            movie_id = None

            for line in f:
                line = line.strip()

                if line.endswith(":"):
                    movie_id = int(line[:-1])
                else:
                    customer_id, rating, date = line.split(",")

                    writer.writerow([
                        int(customer_id),
                        movie_id,
                        int(rating),
                        date
                    ])

# Load CSV

In [4]:
df = pd.read_csv(
    "/kaggle/working/netflix_ratings.csv",
    dtype={
        "customer_id": "int32",
        "movie_id": "int16",
        "rating": "int8"
    })

In [5]:
df.head()

,customer_id,movie_id,rating,date
0,1488844,1,3,2005-09-06
1,822109,1,5,2005-05-13
2,885013,1,4,2005-10-19
3,30878,1,4,2005-12-26
4,823519,1,3,2004-05-03


In [6]:
print(f"Size: {df.size}")
print(f"Shape: {df.shape}")

Size: 401922028
Shape: (100480507, 4)


# Convert IDs to Matrix

Customer IDs are not sequential, so we map them.

In [7]:
df["user_index"] = df["customer_id"].astype("category").cat.codes
df["movie_index"] = df["movie_id"].astype("category").cat.codes

# Create Sparse Matrix

In [8]:
user_movie_matrix = csr_matrix((df["rating"], (df["user_index"], df["movie_index"])))

In [9]:
print(user_movie_matrix.shape)

(480189, 17770)


# Find Global Mean of Movies

In [10]:
# counting non-zero because sparse matrix. This can be only done on sparse matrix
count = user_movie_matrix.nnz
val = user_movie_matrix.sum()
print(f"Sum: {val}")
print(f"Count: {count}")

mu = val/count

print(f"Mean: {mu}")

Sum: 362160883
Count: 100480507
Mean: 3.604289964420661


There are total 10,04,80,507 (\~10 Crore) Entries in the matrix.<br>
Their total sum is 36,21,60,883 (\~36 Crore).<br>
Global Mean of Movies is **3.6 out of 5**

# Initialise Hyperparameters and Parameters

## Initialise Hyperparameters

In [11]:
k = 20
gamma = 0.0005
lam = 0.01

## Initialise Parameters

In [12]:
number_of_users = df["user_index"].nunique()
number_of_movies = df["movie_index"].nunique()

print(f"Number of Users: {number_of_users}")
print(f"Number of Movies: {number_of_movies}")

Number of Users: 480189
Number of Movies: 17770


In [13]:
bu = np.zeros(number_of_users)
bm = np.zeros(number_of_movies)

print(f"Size of User Bias (bu): {len(bu)}")
print(f"Size of Movie Bias (bm): {len(bm)}")

Size of User Bias (bu): 480189
Size of Movie Bias (bm): 17770


In [14]:
rng = np.random.default_rng()

P = rng.normal(0, 0.1, (number_of_users, k))
Q = rng.normal(0, 0.1, (number_of_movies, k))

# Training

$$
\hat{r_{um}} = \mu + b_{u} + b_{m} + P_{u}^{T} \cdot Q_{m}
$$

$$
e = r - \hat{r}
$$

In [34]:
rows, cols = user_movie_matrix.nonzero()
ratings = user_movie_matrix.data
epochs = 20

@njit
def train(rows, cols, ratings, P, Q, bu, bm, mu, gamma, lam, epochs):
    for epoch in range(epochs):
        print(f"Starting Training of Epoch: {epoch}")

        for u, m, r in zip(rows, cols, ratings):
            # print(f"Before Epoch: {epoch}.\nUser Bias: {bu[u]}\nMovie Bias:{bm[m]}\nUser Latent Factor: {P[u]}\nMovie Latent Factor: {Q[m]}")
            r_hat = mu + bu[u] + bm[m] + np.dot(P[u], Q[m])

            e = r - r_hat

            P_old = P[u].copy()
            Q_old = Q[m].copy()

            bu[u] += gamma * (e - lam * bu[u])
            bm[m] += gamma * (e - lam * bm[m])

            P[u] += gamma * (e * Q_old - lam * P[u])
            Q[m] += gamma * (e * P_old - lam * Q[m])
            # print(f"After Epoch: {epoch}.\nUser Bias: {bu[u]}\nMovie Bias:{bm[m]}\nUser Latent Factor: {P[u]}\nMovie Latent Factor: {Q[m]}")

        print(f"Ended Training of Epoch: {epoch}")

train(rows, cols, ratings, P, Q, bu, bm, mu, gamma, lam, epochs)

Starting Training of Epoch: 0
Ended Training of Epoch: 0
Starting Training of Epoch: 1
Ended Training of Epoch: 1
Starting Training of Epoch: 2
Ended Training of Epoch: 2
Starting Training of Epoch: 3
Ended Training of Epoch: 3
Starting Training of Epoch: 4
Ended Training of Epoch: 4
Starting Training of Epoch: 5
Ended Training of Epoch: 5
Starting Training of Epoch: 6
Ended Training of Epoch: 6
Starting Training of Epoch: 7
Ended Training of Epoch: 7
Starting Training of Epoch: 8
Ended Training of Epoch: 8
Starting Training of Epoch: 9
Ended Training of Epoch: 9
Starting Training of Epoch: 10
Ended Training of Epoch: 10
Starting Training of Epoch: 11
Ended Training of Epoch: 11
Starting Training of Epoch: 12
Ended Training of Epoch: 12
Starting Training of Epoch: 13
Ended Training of Epoch: 13
Starting Training of Epoch: 14
Ended Training of Epoch: 14
Starting Training of Epoch: 15
Ended Training of Epoch: 15
Starting Training of Epoch: 16
Ended Training of Epoch: 16
Starting Training

# Compute RMSE

In [35]:
@njit
def rmse(rows, cols, ratings, P, Q, mu, bu, bm):

    sq_error = 0.0
    k = P.shape[1]

    for u, m, r in zip(rows, cols, ratings):
        dot = 0.0
        for f in range(k):
            dot += P[u, f] * Q[m, f]

        pred = mu + bu[u] + bm[m] + dot

        if pred < 1:
            pred = 1
        elif pred > 5:
            pred = 5

        err = r - pred
        sq_error += err * err

    return np.sqrt(sq_error / len(ratings))

In [36]:
rmse_score = rmse(rows, cols, ratings, P, Q, mu, bu, bm)

rmse_score

0.8723116531761097

The RMSE is 0.92 at 10 epoch.<br>
The RMSE is 0.87 at 20 epoch.<br>
Netflix's CineMatch RMSE Score was 0.95

# Testing

## Testing against Probe

In [42]:
# Creating the Lookup Table
user_map = dict(zip(df["customer_id"], df["user_index"]))
movie_map = dict(zip(df["movie_id"], df["movie_index"]))

In [43]:
probe_users = []
probe_movies = []
probe_ratings = []

movie = None

with open("/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/probe.txt", "r") as f:
    for line in f:
        line = line.strip()

        if line.endswith(":"):
            movie_id = int(line[:-1])
            movie = movie_map[movie_id]
        else:
            user_id = int(line)
            if user_id in user_map:
                user = user_map[user_id]
                probe_users.append(user)
                probe_movies.append(movie)
                rating = user_movie_matrix[user, movie]
                probe_ratings.append(rating)

probe_users = np.array(probe_users, dtype=np.int32)
probe_movies = np.array(probe_movies, dtype=np.int32)
probe_ratings = np.array(probe_ratings, dtype=np.int32)

In [50]:
print(len(probe_users))
print(len(probe_movies))
print(len(probe_ratings))

1408395
1408395
1408395


In [59]:
probe_rmse = rmse(probe_users, probe_movies, probe_ratings, P, Q, mu, bu, bm)

probe_rmse

0.9473386285945342

RMSE is 1.002 at 10 epoch<br>
RMSE is 0.94 at 20 epoch<br>
Netflix's CineMatch RMSE Score was 0.95

## Predicting on Qualifying

In [54]:
qual_users = []
qual_movies = []

movie = None

with open("/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/qualifying.txt") as f:

    for line in f:

        line = line.strip()

        if line.endswith(":"):

            movie_id = int(line[:-1])
            movie = movie_map[movie_id]

        else:

            user_id, _ = line.split(",")

            user_id = int(user_id)

            if user_id in user_map:

                user = user_map[user_id]

                qual_users.append(user)
                qual_movies.append(movie)

qual_users = np.array(qual_users, dtype=np.int32)
qual_movies = np.array(qual_movies, dtype=np.int32)

predictions = []

for u, m in zip(qual_users, qual_movies):

    pred = mu + bu[u] + bm[m] + np.dot(P[u], Q[m])

    pred = np.clip(pred, 1, 5)

    predictions.append(pred)

predictions = np.array(predictions, dtype=np.float32)

# Adding Moive Names

In [63]:
import chardet

# Open the file in Binary Mode ('rb')
with open('/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/movie_titles.csv', 'rb') as f:
    raw_data = f.read(10000)  # Read the first 10k bytes for a good sample
    result = chardet.detect(raw_data)
    
print(f"Detected Encoding: {result['encoding']}")
print(f"Confidence: {result['confidence'] * 100}%")

Detected Encoding: ascii
Confidence: 100.0%


In [70]:
movies_data = []

with open("/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/movie_titles.csv", encoding="latin1") as f:

    for line in f:

        parts = line.strip().split(",", 2)  # split only first 2 commas

        movie_id = int(parts[0])
        year = parts[1]
        title = parts[2]

        movies_data.append((movie_id, year, title))

movie_names = pd.DataFrame(movies_data, columns=["movie_id", "year", "title"])

movie_names["movie_index"] = movie_names["movie_id"].map(movie_map)

movie_names = movie_names.sort_values("movie_index")

movie_titles = movie_names["title"].values
movie_years = movie_names["year"].values

In [74]:
np.savez(
    "netflix.npz",
    P=P,
    Q=Q,
    bu=bu,
    bm=bm,
    mu=mu,
    movie_titles = movie_titles,
    movie_years = movie_years
)